# arc3-cwm-prototype - CodeWorldAgent on the served model

Free run, **no submission quota**. Setup cells are reused verbatim from `arc3-duck-nvfp4-anim` (itself a reproduction of other people's work -- see that notebook's `THIRD_PARTY_NOTICE.md`) purely to boot the NVFP4 vLLM server; the games are then played by our own `CodeWorldAgent`, not by the duck solver.

Changes since the 2026-09-09 live test, which scored 0/25 replay passes with Qwen3-Coder-30B over `transformers`:

* the served model is **Qwen3.8-Flash-Next-NVFP4** behind vLLM;
* reasoning is **disabled** (measured: with it on, 27/27 replies hit the token cap and none contained a `class WorldModel`);
* the transcript now **resets per level**, so clearing a level no longer makes the replay gate permanently unsatisfiable.


In [ ]:
# [calamitychasm] ADDED FOR THIS FORK -- diagnostic only, changes no behaviour.
# The upstream README warns that a manual copy must select the RTX PRO 6000 by hand.
# We push via the API with machine_shape=NvidiaRtxPro6000, which IS honoured (verified
# in experiments/stage7_duck_nvfp4.md), but a wrong card would waste the whole run, so
# print what we actually got before anything expensive happens.
import os, shutil, subprocess

print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap,driver_version",
     "--format=csv,noheader"],
    capture_output=True, text=True).stdout.strip() or "nvidia-smi unavailable")
try:
    _mem_kb = int(next(l.split()[1] for l in open("/proc/meminfo") if l.startswith("MemTotal")))
    _ram = f"{_mem_kb / 1048576:.1f}"
except Exception:
    _ram = "?"
print(f"HW_PROBE host_ram_gib={_ram}  cpu_count={os.cpu_count()}  "
      f"free_disk_gib={shutil.disk_usage('/kaggle/working').free / 2**30:.1f}  "
      f"rerun={os.getenv('KAGGLE_IS_COMPETITION_RERUN')!r}")


## 1. Environment and submission mode

Detect whether this is a real competition rerun (which minimises diagnostics), set the
framework's environment flags, and put the CUDA libraries on the linker path.

In [ ]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# Skip periodic JSON/HTML diagnostics and per-frame logging for every run.
# thui-animfast: full diagnostics on an interactive public run (usage/events/transcript sidecars); minimal in a rerun.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"

# Apply the measured vLLM winner before any serving setup command runs.
PUBLIC25_VLLM_PROFILE_NAME = 'kv5-bf16-mtp3-c8-cg32'
PUBLIC25_VLLM_PROFILE_ENV = {
    "TAAF_VLLM_ENABLE_PREFIX_CACHING": "0",
    "TAAF_VLLM_KV_CACHE_DTYPE": "auto",
    "TAAF_VLLM_KV_CACHE_MEMORY_BYTES": "5368709120",
    "TAAF_VLLM_MAX_CUDAGRAPH_CAPTURE_SIZE": "32",
    "TAAF_VLLM_MAX_NUM_BATCHED_TOKENS": "8192",
    "TAAF_VLLM_MAX_NUM_SEQS": "8",
    "TAAF_VLLM_MTP_TOKENS": "3",
    "TAAF_VLLM_OMP_THREADS": "1"
}
for key, value in PUBLIC25_VLLM_PROFILE_ENV.items():
    os.environ[key] = value
print(
    f'PUBLIC25_VLLM_PROFILE name={PUBLIC25_VLLM_PROFILE_NAME} '
    f'env={json.dumps(PUBLIC25_VLLM_PROFILE_ENV, sort_keys=True)}',
    flush=True,
)
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Prepend the CUDA toolkit to the linker path (it is off it on Kaggle GPU images) so the
# solver's GPU libraries (e.g. vllm / torch) can link against libcuda.
cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

# Everything the run produces is written here.
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")

## 2. Install the ARC runtime

Install `arc-agi` from the offline competition wheelhouse (the Kaggle submission environment
has no internet).

In [ ]:
# thui-animfast: resolve the competition mount instead of assuming its layout -- Kaggle serves either
# /kaggle/input/competitions/<comp> or /kaggle/input/<comp>, and which one varies between runs.
_COMP_CANDIDATES = ["/kaggle/input/competitions/arc-prize-2026-arc-agi-3", "/kaggle/input/arc-prize-2026-arc-agi-3"]
_COMP_DIR = next((_p for _p in _COMP_CANDIDATES if os.path.isdir(_p)), None)
assert _COMP_DIR is not None, (
    "thui-animfast: no competition mount found. Tried " + repr(_COMP_CANDIDATES)
    + "; /kaggle/input holds "
    + repr(sorted(os.listdir("/kaggle/input")) if os.path.isdir("/kaggle/input") else "MISSING")
)
_WHEELS = os.path.join(_COMP_DIR, "arc_agi_3_wheels")
assert os.path.isdir(_WHEELS), "thui-animfast: resolved wheels dir is not a directory: " + _WHEELS
print("thui-animfast: competition mount = " + _COMP_DIR, flush=True)
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        _WHEELS,
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)

## 3. Locate the source bundle

Find the uploaded TAAF source dataset by its marker file, and record where Kaggle mounted
every attached input so setup commands and the solver can find them.

In [ ]:
# Kaggle inputs attached to this notebook, plus bookkeeping paths used below.
DATASET_SOURCES = ["keithtyser/duck-qwen38-nvfp4-mtp-vllm-smoke-v1", "keithtyser/qwen38-flash-next-vllm-nvfp4-runtime-v1", "jakobbrggen/taaf-kaggle-source-anim-20260807-anim"]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


# Locate the source dataset by its marker file rather than a fixed mount path.
def _find_bundle_dir(label: str) -> Path:
    # thui-animfast: TWO attached datasets carry the marker (his June duck bundle and the anim bundle), so
    # "first marker wins" is a coin flip -- pick by the benchmark_label the marker file records.
    found = {}
    for marker in Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER):
        try:
            found[json.loads(marker.read_text())["benchmark_label"]] = marker.parent
        except Exception as exc:
            print(f"thui-animfast: unreadable marker {marker}: {exc!r}", flush=True)
    if label not in found:
        raise RuntimeError(f"TAAF source bundle {label!r} not found under /kaggle/input; markers = {found}")
    return found[label]


# Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
# (depending on owner / slug collisions), so probe both and use whichever exists. Utility
# scripts mount under /kaggle/usr/lib/notebooks/<owner>/<slug>.
def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((c for c in candidates if c.exists()), None)


BUNDLE_DIR = _find_bundle_dir("duck-harness-kaggle")          # his: serving_setup.py, vllm patches, watchdog, teardown
ANIM_BUNDLE_DIR = _find_bundle_dir("anim-20260807-anim")     # ours: the solver tree + its pickled benchmark / target
assert BUNDLE_DIR != ANIM_BUNDLE_DIR, "thui-animfast: both labels resolved to one directory"
assert (BUNDLE_DIR / "serving_setup.py").is_file(), f"thui-animfast: his bundle has no serving_setup.py: {BUNDLE_DIR}"
assert (ANIM_BUNDLE_DIR / "src" / "ARC3-Inference" / "inference" / "utils" / "animation.py").is_file(), (
    f"thui-animfast: the anim bundle has no animation.py: {ANIM_BUNDLE_DIR}")
print(f"thui-animfast: anim bundle = {ANIM_BUNDLE_DIR}", flush=True)
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")

# Map each attached input to where Kaggle actually mounted it (the source bundle is index 0).
kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# Published to setup commands and the solver via the environment:
setup_env = {
    # JSON {ref: mount_path} so they can locate every attached dataset / utility script.
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    # The attached dataset refs in order (index 0 is this source bundle).
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    # The attached utility-script / kernel refs.
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

## 4. Import the bundled source and run solver setup

Put the snapshotted repositories on the path (this process and any child processes), then run
the solver's setup commands â€” installing wheels, fetching model weights, and so on.

In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
# thui-animfast: his tree minus the two solver repos (the June duck), plus the anim solver tree. The loop below
# inserts each entry at sys.path[0], so the LAST entries win -- the anim ones; the .pth is written anim-first.
_SOLVER_REPOS = {"ARC3-Inference", "tufa-arc-agi-framework"}
_his_entries = [e for e in _source_path_entries(BUNDLE_DIR) if e.parent.name not in _SOLVER_REPOS and e.name not in _SOLVER_REPOS]
_anim_entries = _source_path_entries(ANIM_BUNDLE_DIR)
assert _anim_entries and all(str(e).startswith(str(ANIM_BUNDLE_DIR)) for e in _anim_entries), _anim_entries
assert not any(("ARC3-Inference" in str(e) or "tufa-arc-agi-framework" in str(e)) for e in _his_entries), _his_entries
source_entries = _his_entries + _anim_entries
print(f"thui-animfast: source roots his={[str(e) for e in _his_entries]} anim={[str(e) for e in _anim_entries]}", flush=True)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in (_anim_entries + _his_entries)))   # anim first for child processes
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

# Solver setup commands (wheels, vLLM server startup, ...) run before the benchmark loads.
env = _command_env()
for command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
    print(f"taaf.kaggle: setup command: {command}", flush=True)
    subprocess.run(command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    # Re-read in case the command persisted new env keys.
    env = _command_env()
    os.environ.update(env)

# Honour any PYTHONPATH a setup command exported.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)
# ---- thui-animfast: the thui-v3 knobs, set AFTER his serving_setup persisted the analyzer env and BEFORE any
# `inference` import (tool_agent reads LOCAL_ANALYZER_SEED / YIELD_SECONDS at import time), then the graft teeth.
_KNOBS = {"LOCAL_ANALYZER_SEED": "20260825", "LOCAL_ANALYZER_YIELD_SECONDS": "180"}
_persisted = json.loads(SETUP_ENV_PATH.read_text())
assert _persisted.get("LOCAL_ANALYZER_MODEL_ID") == "Qwen/Qwen3.8-Flash-Next-NVFP4", _persisted.get("LOCAL_ANALYZER_MODEL_ID")
assert _persisted.get("LOCAL_ANALYZER_YIELD_SECONDS") == "60", "his serving_setup no longer persists yield 60 -- re-derive the override"
assert _persisted.get("LOCAL_ANALYZER_TEMPERATURE") == "0.6" and _persisted.get("MULTIMODAL_UPSCALE") == "4", _persisted
_persisted.update(_KNOBS)
SETUP_ENV_PATH.write_text(json.dumps(_persisted, indent=2, sort_keys=True) + "\n")
os.environ.update(_KNOBS)
assert "inference" not in sys.modules and "taaf" not in sys.modules, "solver imported before the knob override"
import inference.agent.tool_agent as _tool_agent
import inference.utils.animation as _anim_mod
import taaf as _taaf
for _m in (_tool_agent, _anim_mod, _taaf):
    assert str(Path(_m.__file__).resolve()).startswith(str(ANIM_BUNDLE_DIR.resolve())), (_m.__name__, _m.__file__)
assert _tool_agent._LOCAL_ANALYZER_SEED == int("20260825"), _tool_agent._LOCAL_ANALYZER_SEED
assert float(_tool_agent._LOCAL_ANALYZER_YIELD_SECONDS) == float("180"), _tool_agent._LOCAL_ANALYZER_YIELD_SECONDS
assert os.environ["LOCAL_ANALYZER_MODEL_ID"] == "Qwen/Qwen3.8-Flash-Next-NVFP4"
print(f"THUI_ANIMFAST_GRAFT ok solver={Path(_tool_agent.__file__).parent} seed={_tool_agent._LOCAL_ANALYZER_SEED} "
      f"yield={_tool_agent._LOCAL_ANALYZER_YIELD_SECONDS} model={os.environ['LOCAL_ANALYZER_MODEL_ID']} "
      f"temperature={os.environ['LOCAL_ANALYZER_TEMPERATURE']} upscale={os.environ['MULTIMODAL_UPSCALE']}", flush=True)


In [ ]:
!pip install --no-index --find-links /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels arc-agi python-dotenv


In [ ]:
# ---------------------------------------------------------------------------
# CodeWorldAgent LIVE DIAGNOSTIC -- runs UNCONDITIONALLY (no
# KAGGLE_IS_COMPETITION_RERUN gate). That gate is exactly why every previous
# free push validated nothing: it hides all the real setup + agent code.
#
# THIS KERNEL IS NEVER SUBMITTED FOR SCORING. It exists only to answer:
# can Qwen3-Coder-30B-A3B write a replay-passing WorldModel for a real game?
# ---------------------------------------------------------------------------
import base64
import glob
import os
import shutil
import subprocess
import sys
import time

_t0 = time.time()
def _el():
    return f"{time.time() - _t0:6.1f}s"


def find_model_dir(keyword):
    """Same resolution the real submission notebook uses."""
    candidates = []
    for path in glob.glob("/kaggle/input/**/config.json", recursive=True):
        if keyword.lower() in path.lower():
            candidates.append(os.path.dirname(path))
    if not candidates:
        return None
    return sorted(candidates)[-1]


CODER_MODEL_DIR = find_model_dir("qwen3-coder") or find_model_dir("qwen")
GEMMA_MODEL_DIR = find_model_dir("gemma-3-12b-it") or find_model_dir("gemma")
print(f"[{_el()}] CODER_MODEL_DIR = {CODER_MODEL_DIR}", flush=True)
print(f"[{_el()}] GEMMA_MODEL_DIR = {GEMMA_MODEL_DIR}  (resolved for parity; NOT loaded)", flush=True)
# CODER_MODEL_DIR is unused on the served path (LLM_BACKEND=openai).
print("CODER_MODEL_DIR (unused, served path):", CODER_MODEL_DIR)

print(f"[{_el()}] === copying competition harness + environment_files ===", flush=True)
_COMP = "/kaggle/input/competitions/arc-prize-2026-arc-agi-3"
if not os.path.exists("/kaggle/working/ARC-AGI-3-Agents"):
    shutil.copytree(
        f"{_COMP}/ARC-AGI-3-Agents",
        "/kaggle/working/ARC-AGI-3-Agents",
        ignore=shutil.ignore_patterns(".git"),
    )
    shutil.copytree(
        f"{_COMP}/environment_files",
        "/kaggle/working/ARC-AGI-3-Agents/environment_files",
    )
_n_envs = len(glob.glob("/kaggle/working/ARC-AGI-3-Agents/environment_files/**/metadata.json", recursive=True))
print(f"[{_el()}] environment_files: {_n_envs} metadata.json found", flush=True)

print(f"[{_el()}] === copying llm_engine + agent from the FIXED dataset ===", flush=True)
_DATASET = "/kaggle/input/datasets/calamitychasm/llm-world-engine-agent-fixed"
print(f"[{_el()}] dataset mount exists: {os.path.exists(_DATASET)}", flush=True)
if not os.path.exists(_DATASET):
    # Dump the real tree rather than guessing at the mount convention.
    for root, dirs, files in os.walk("/kaggle/input"):
        if root.count("/") <= 5:
            print("   ", root, dirs[:8], files[:8], flush=True)
    raise SystemExit("dataset mount path not found -- see tree above")

if os.path.exists("/kaggle/working/llm_engine"):
    shutil.rmtree("/kaggle/working/llm_engine")
shutil.copytree(f"{_DATASET}/llm_engine", "/kaggle/working/llm_engine")
shutil.copy(
    f"{_DATASET}/code_world_agent.py",
    "/kaggle/working/ARC-AGI-3-Agents/agents/templates/code_world_agent.py",
)

with open("/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py", "w") as f:
    f.write(
        "from typing import Type, cast\n"
        "from dotenv import load_dotenv\n"
        "from .agent import Agent, Playback\n"
        "from .swarm import Swarm\n"
        "from .templates.random_agent import Random\n"
        "from .templates.code_world_agent import CodeWorldAgent\n"
        "\n"
        "load_dotenv()\n"
        "\n"
        "AVAILABLE_AGENTS: dict[str, Type[Agent]] = {\n"
        "    \"random\": Random,\n"
        "    \"codeworldagent\": CodeWorldAgent,\n"
        "}\n"
    )

with open("/kaggle/working/ARC-AGI-3-Agents/.env", "w") as f:
    f.write(
        "ARC_API_KEY=offline-diag\n"
        "OPERATION_MODE=offline\n"
        "ENVIRONMENTS_DIR=/kaggle/working/ARC-AGI-3-Agents/environment_files\n"
        "RECORDINGS_DIR=/kaggle/working/diag_recordings\n"
    )

print(f"[{_el()}] === writing diag driver ===", flush=True)
DRIVER_B64 = "..."
with open("/kaggle/working/diag_driver.py", "wb") as f:
    f.write(base64.b64decode(DRIVER_B64))

run_env = {
    **os.environ,
    "MPLBACKEND": "agg",
    "LLM_BACKEND": "openai",
    "CODER_LLM_BASE_URL": "http://127.0.0.1:1234/v1",
    "CODER_LLM_MODEL": "Qwen/Qwen3.8-Flash-Next-NVFP4",
    "ACTION_LLM_BASE_URL": "http://127.0.0.1:1234/v1",
    "ACTION_LLM_MODEL": "Qwen/Qwen3.8-Flash-Next-NVFP4",
    "LLM_ENABLE_THINKING": "0",
    "CODER_MODEL_DIR": CODER_MODEL_DIR,
    # The action head is budget-disabled in the driver; pointing it at the
    # coder dir means get_shared_transformers_client() hands back the same
    # already-loaded model instead of putting a second multi-GB model in
    # VRAM alongside the 30B coder.
    "ACTION_MODEL_DIR": CODER_MODEL_DIR,
    "GEMMA_MODEL_DIR_PARITY": GEMMA_MODEL_DIR or "",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "ARC_API_KEY": "offline-diag",
    "OPERATION_MODE": "offline",
    "ENVIRONMENTS_DIR": "/kaggle/working/ARC-AGI-3-Agents/environment_files",
    "RECORDINGS_DIR": "/kaggle/working/diag_recordings",
    "DIAG_N_GAMES": "12",
    "DIAG_MAX_ACTIONS": "120",
    "DIAG_CODER_BUDGET": "8",
    "DIAG_DRAFT_ATTEMPTS": "5",
    "DIAG_REPAIR_ATTEMPTS": "2",
    "DIAG_LLM_DEADLINE_MIN": "150",
    "DIAG_RUN_DEADLINE_MIN": "180",
    "PYTHONUNBUFFERED": "1",
}

print(f"[{_el()}] === running diag driver ===", flush=True)
result = subprocess.run(
    [sys.executable, "-u", "/kaggle/working/diag_driver.py"],
    cwd="/kaggle/working/ARC-AGI-3-Agents",
    env=run_env,
)
print(f"[{_el()}] === diag driver exited with code {result.returncode} ===", flush=True)


In [ ]:
# Surface the machine-readable evidence in the notebook output too, so it
# survives even if the output file download is unavailable.
import json
import os

p = "/kaggle/working/diag_evidence.json"
if os.path.exists(p):
    ev = json.load(open(p))
    print("SUMMARY:", json.dumps(ev.get("summary"), indent=2))
    print("FIXED-CODE CHECKS:", json.dumps(ev.get("fixed_code_checks"), indent=2))
    print("ROUNDS:", json.dumps(ev.get("rounds"), indent=2))
    print("REPLAY RESULTS:", json.dumps(ev.get("replay_results"), indent=2))
    print("LOAD RESULTS:", json.dumps(
        [{k: v for k, v in r.items()} for r in ev.get("load_results", [])], indent=2))
    print("GAMES:", json.dumps(
        [{k: v for k, v in g.items() if k != "model_source"} for g in ev.get("games", [])],
        indent=2))
    print("ERRORS:", json.dumps(ev.get("errors"), indent=2))
else:
    print("NO EVIDENCE FILE at", p)

# Also drop the world_models/ revision trail into the output.
for root, dirs, files in os.walk("/kaggle/working/world_models"):
    for fn in files:
        print("revision artifact:", os.path.join(root, fn))
